# 03_build_index.ipynb

Chunked FAISS embedding index build.
Only the embedding/index creation part is changed to avoid RAM overflow.
Project imports and preprocessing follow the existing notebooks.


In [4]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
import gc
import glob

import pandas as pd
import numpy as np
import faiss

from src.rag.config import load_config
from src.rag.utils.text import build_comment_text
from src.rag.preprocessing.processor import TextProcessor
from src.rag.embedding.factory import EmbeddingFactory


In [5]:
config = load_config(
    "../configs/rag.yaml"
)

config


{'retrieval': {'type': 'hybrid',
  'top_k': 5,
  'hybrid': {'bm25_weight': 0.3, 'embedding_weight': 0.7}},
 'embedding': {'provider': 'sentence_transformer',
  'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}}

In [6]:
comments = pd.read_parquet(
    "../data/processed/comments_clean.parquet"
)

comments.head()


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate,created_at_gregorian
0,14144758,هری پاتر,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,10 آذر 1399,5.0,recommended,True,1075274,NaN,NaN,1136,8,بیگای استودیو,5AADE,NaN,2020-11-30
1,41782279,روغن ریش,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,21 آبان 1401,3.0,recommended,True,6081008,NaN,NaN,896,79,گراندو بیوتی,6XN5S,NaN,2022-11-12
2,49569443,تقلبی,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,3 خرداد 1402,1.0,not_recommended,True,10545754,NaN,NaN,625,60,کافه سرگرمی,AXVST,NaN,2023-05-24
3,43932524,یک نظر بی اغراق!,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,15 دی 1401,5.0,recommended,True,4153832,NaN,NaN,524,51,گروه آروند,5AD65,NaN,2023-01-05
4,21396693,NaN,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,9 خرداد 1400,3.0,no_idea,True,2185657,NaN,NaN,450,2,تاباتا,C93UM,NaN,2021-05-30


In [7]:
processor = TextProcessor()


comments["search_text"] = (
    comments
    .apply(build_comment_text, axis=1)
    .apply(processor.process)
)


documents = comments[
    [
        "id",
        "product_id",
        "body",
        "rate",
        "search_text"
    ]
].copy()


documents.head()


,id,product_id,body,rate,search_text
0,14144758,1075274,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,5.0,هری پاتر عالیه مخصوصا طرحش عکس مجموعه هری پاتر...
1,41782279,6081008,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,3.0,روغن ریش توجه فرمایید که عکس اول واسه ۲ هفته پ...
2,49569443,10545754,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,1.0,تقلبی متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.ب...
3,43932524,4153832,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,5.0,یک نظر بی اغراق! تصمیم خرید کنسول برای منِ 32 ...
4,21396693,2185657,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,3.0,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...


In [8]:
embedding_model = EmbeddingFactory.create(
    config["embedding"]["provider"],
    config["embedding"]["model"]
)

embedding_model


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8621.70it/s]


In [10]:
index_dir = Path(
    "../data/indexes/product_comments_embedding"
)

chunk_dir = index_dir / "chunks"

chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)


batch_size = 5000
encode_batch_size = 64


texts = (
    documents["search_text"]
    .fillna("")
    .astype(str)
    .tolist()
)


In [11]:
for start in range(
    0,
    len(texts),
    batch_size
):

    end = min(
        start + batch_size,
        len(texts)
    )

    print(
        f"Encoding {start}:{end}"
    )


    embeddings = embedding_model.encode(
        texts[start:end],
        device="cuda",
        batch_size=encode_batch_size
    )


    embeddings = np.asarray(
        embeddings
    ).astype("float32")


    faiss.normalize_L2(
        embeddings
    )


    chunk_index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    chunk_index.add(
        embeddings
    )


    faiss.write_index(
        chunk_index,
        str(chunk_dir / f"chunk_{start}.faiss")
    )


    del embeddings
    del chunk_index

    gc.collect()


print("All chunks saved")


Encoding 0:5000
Encoding 5000:10000
Encoding 10000:15000
Encoding 15000:20000
Encoding 20000:25000
Encoding 25000:30000
Encoding 30000:35000
Encoding 35000:40000
Encoding 40000:45000
Encoding 45000:50000
Encoding 50000:55000
Encoding 55000:60000
Encoding 60000:65000
Encoding 65000:70000
Encoding 70000:75000
Encoding 75000:80000
Encoding 80000:85000
Encoding 85000:90000
Encoding 90000:95000
Encoding 95000:100000
Encoding 100000:105000
Encoding 105000:110000
Encoding 110000:115000
Encoding 115000:120000
Encoding 120000:125000
Encoding 125000:130000
Encoding 130000:135000
Encoding 135000:140000
Encoding 140000:145000
Encoding 145000:150000
Encoding 150000:155000
Encoding 155000:160000
Encoding 160000:165000
Encoding 165000:170000
Encoding 170000:175000
Encoding 175000:180000
Encoding 180000:185000
Encoding 185000:190000
Encoding 190000:195000
Encoding 195000:200000
Encoding 200000:205000
Encoding 205000:210000
Encoding 210000:215000
Encoding 215000:220000
Encoding 220000:225000
Encoding 2

In [12]:
documents.to_parquet(
    index_dir / "metadata.parquet",
    index=False
)

print(
    "Metadata saved:",
    index_dir / "metadata.parquet"
)

print(
    "Rows:",
    len(documents)
)

Metadata saved: ../data/indexes/product_comments_embedding/metadata.parquet
Rows: 6153060
